# Creación del Dataset Final - Contaminación, Tráfico y Meteorología en Madrid

En este notebook unimos los 5 ficheros originales para construir el dataset que usaremos en el EDA y el modelado.

**Target**: NO2 (nivel de contaminación)

**Features**: variables de tráfico y meteorología

**Pasos:**
1. Cargar los 5 CSV originales
2. Pasar calidad del aire y meteo de formato ancho a formato largo
3. Corregir la escala de las variables meteorológicas (problema de decimales)
4. Emparejar geográficamente estaciones de aire con tráfico y meteo cuando no comparten código
5. Unir todo y quedarnos con las columnas que nos interesan
6. Guardar el dataset final

## 1. Cargar los datos

In [15]:
import pandas as pd
import numpy as np

ruta = "../data_sample/"  # ajusta esta ruta segun donde tengas tu notebook

calidad_aire = pd.read_csv(ruta + "calidad_del_aire.csv", sep=";", encoding="utf-8-sig")
meteo_raw = pd.read_csv(ruta + "datos_meteorologicos.csv", sep=";", encoding="utf-8-sig")
estaciones_aire = pd.read_csv(ruta + "estaciones_control_aire.csv", sep=";", encoding="utf-8-sig")
estaciones_trafico = pd.read_csv(ruta + "estaciones_medicion_trafico.csv", sep=";", encoding="latin1")
# NOTA: el fichero trafico_diario.csv tiene un problema conocido en su cabecera
# (texto "git status" pegado por error delante de "fecha"). Por eso cargamos sin
# parse_dates y renombramos la primera columna manualmente, para que funcione
# sin importar el nombre exacto que tenga esa primera columna.
trafico = pd.read_csv(ruta + "trafico_diario.csv", sep=";", encoding="latin1")
trafico = trafico.rename(columns={trafico.columns[0]: "fecha"})
trafico["fecha"] = pd.to_datetime(trafico["fecha"])

print("Calidad del aire:", calidad_aire.shape)
print("Meteo:", meteo_raw.shape)
print("Estaciones aire:", estaciones_aire.shape)
print("Estaciones trafico:", estaciones_trafico.shape)
print("Trafico diario:", trafico.shape)

Calidad del aire: (9831, 69)
Meteo: (190564, 56)
Estaciones aire: (24, 25)
Estaciones trafico: (4962, 9)
Trafico diario: (5710123, 12)


In [16]:
prueba = pd.read_csv(ruta + "trafico_diario.csv", sep=";", nrows=3)
print(prueba.columns.tolist())
prueba

['git statusfecha', 'id_estacion_trafico', 'intensidad_mean', 'intensidad_max', 'ocupacion_mean', 'ocupacion_max', 'carga_mean', 'carga_max', 'vmed_mean', 'vmed_max', 'tipo_elem_moda', 'periodo_integracion_moda']


,git statusfecha,id_estacion_trafico,intensidad_mean,intensidad_max,ocupacion_mean,ocupacion_max,carga_mean,carga_max,vmed_mean,vmed_max,tipo_elem_moda,periodo_integracion_moda
0,2019-01-01,1001,2103.125,3096,8.489583,12.0,0.0,0,60.864583,65.0,M30,5
1,2019-01-01,1002,1402.750,3156,5.812500,13.0,0.0,0,68.072917,78.0,M30,5
2,2019-01-01,1003,1812.500,4248,5.854167,13.0,0.0,0,72.645833,93.0,M30,5


## 2. Calidad del aire: de formato ancho a formato largo

El fichero original tiene una columna por cada día del mes (D01, D02...D31) y su columna de validez (V01, V02...V31). 
Nos quedamos con el NO2 (código de magnitud = 8) y pasamos a tener una fila por estación y día.

In [17]:
# Nos quedamos solo con NO2 (magnitud 8)
no2 = calidad_aire[calidad_aire["MAGNITUD"] == 8].copy()

registros_no2 = []

for _, fila in no2.iterrows():
    for dia in range(1, 32):
        col_dato = f"D{dia:02d}"
        col_validez = f"V{dia:02d}"

        if col_dato not in fila or pd.isna(fila[col_dato]):
            continue
        if fila[col_validez] != "V":  # descartamos los no validos
            continue
        try:
            fecha = pd.Timestamp(year=int(fila["ANO"]), month=int(fila["MES"]), day=dia)
        except ValueError:
            continue  # dias que no existen en ese mes (ej 30 de febrero)

        registros_no2.append({
            "estacion": fila["ESTACION"],
            "fecha": fecha,
            "no2": fila[col_dato]
        })

no2_largo = pd.DataFrame(registros_no2)
print("NO2 en formato largo:", no2_largo.shape)
no2_largo.head()

NO2 en formato largo: (51001, 3)


,estacion,fecha,no2
0,4,2021-01-01,10
1,4,2021-01-02,23
2,4,2021-01-03,29
3,4,2021-01-04,29
4,4,2021-01-05,54


## 3. Meteorología: de formato ancho a formato largo + corrección de escala

Igual que antes, pero con horas (H01...H24) en vez de días. Calculamos la media diaria de cada variable.

**Importante - problema de datos detectado**: al comparar con la documentación oficial de datos.madrid.es, vimos que la temperatura, la velocidad del viento y la precipitacion habian perdido su punto decimal al exportar el fichero (ej. 12.9ºC aparecia como 129). Hay que dividir esas 3 variables entre 10 para corregirlo. La humedad y la presion ya estan en su escala correcta.

In [18]:
# Codigos de magnitud meteorologica que nos interesan
magnitudes_meteo = {83: "temperatura", 86: "humedad", 81: "viento_vel", 89: "precipitacion"}

meteo = meteo_raw[meteo_raw["MAGNITUD"].isin(magnitudes_meteo.keys())].copy()

registros_meteo = []

for _, fila in meteo.iterrows():
    valores_validos = []
    for h in range(1, 25):
        col_dato = f"H{h:02d}"
        col_validez = f"V{h:02d}"
        if col_validez in fila and fila[col_validez] == "V" and not pd.isna(fila[col_dato]):
            valores_validos.append(fila[col_dato])

    if not valores_validos:
        continue

    try:
        fecha = pd.Timestamp(year=int(fila["ANO"]), month=int(fila["MES"]), day=int(fila["DIA"]))
    except ValueError:
        continue

    registros_meteo.append({
        "estacion": fila["ESTACION"],
        "fecha": fecha,
        "magnitud": magnitudes_meteo[fila["MAGNITUD"]],
        "valor_medio_dia": np.mean(valores_validos)
    })

meteo_largo = pd.DataFrame(registros_meteo)

# Pivotamos para tener una columna por variable meteo
meteo_diario = meteo_largo.pivot_table(index=["estacion", "fecha"], columns="magnitud", values="valor_medio_dia").reset_index()

# Corregimos la escala (dividir entre 10)
meteo_diario["temperatura"] = meteo_diario["temperatura"] / 10
meteo_diario["viento_vel"] = meteo_diario["viento_vel"] / 10
meteo_diario["precipitacion"] = meteo_diario["precipitacion"] / 10

print("Meteo diaria:", meteo_diario.shape)
meteo_diario.describe()

Meteo diaria: (52712, 6)


magnitud,estacion,fecha,humedad,precipitacion,temperatura,viento_vel
count,52712.000000,52712,46390.000000,20751.000000,48412.000000,20553.000000
mean,69.353620,2021-12-21 04:12:54.745788160,55.296864,0.249373,15.362778,12.061642
min,4.000000,2019-01-01 00:00:00,0.000000,0.000000,-9.341667,0.020833
25%,36.000000,2020-06-05 00:00:00,39.000000,0.000000,9.362500,7.166667
50%,59.000000,2021-11-27 00:00:00,54.958333,0.000000,14.272373,10.431818
75%,108.000000,2023-07-31 00:00:00,70.791667,0.000000,21.333333,15.108333
max,115.000000,2024-12-31 00:00:00,1030.928571,22.795833,70.000000,1632.757143
std,39.374874,NaN,20.112194,1.104408,7.583599,13.334947


## 4. Emparejamiento geográfico

**El problema**: las estaciones de aire, de tráfico y de meteo están en sitios físicos distintos de Madrid. No podemos unir tablas por "nombre de estación" porque no coinciden.

**La solución**: usamos las coordenadas (latitud/longitud) de cada punto y calculamos la distancia real en el mapa entre ellos, para asignar a cada estación de aire su punto de tráfico y su estación meteo más cercanos.



In [19]:
# Esta funcion calcula la distancia en km entre dos puntos del mapa, dadas sus coordenadas.

def haversine(lon1, lat1, lon2, lat2):
    # paso 1: pasamos los grados a radianes (lo que necesita la formula matematica)
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    # paso 2: calculamos la diferencia entre las dos coordenadas
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    # paso 3: formula de la distancia sobre una esfera (la Tierra)
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    # paso 4: 6371 es el radio de la Tierra en km, lo usamos para convertir el resultado a km
    return 6371 * 2 * np.arcsin(np.sqrt(a))

In [20]:

estaciones_con_meteo = [4, 8, 16, 18, 24, 35, 36, 38, 39, 54, 56, 58, 59]

aire_con_meteo = estaciones_aire[estaciones_aire["CODIGO_CORTO"].isin(estaciones_con_meteo)]
aire_sin_meteo = estaciones_aire[~estaciones_aire["CODIGO_CORTO"].isin(estaciones_con_meteo)]


mapa_meteo = {e: e for e in estaciones_con_meteo}  # las que ya tienen, se asignan a si mismas
distancia_meteo = {e: 0.0 for e in estaciones_con_meteo}  # distancia 0 porque es la misma estacion

for _, fila in aire_sin_meteo.iterrows():
    distancias = haversine(fila["LONGITUD"], fila["LATITUD"], aire_con_meteo["LONGITUD"], aire_con_meteo["LATITUD"])
    estacion_mas_cercana = aire_con_meteo.loc[distancias.idxmin(), "CODIGO_CORTO"]
    mapa_meteo[fila["CODIGO_CORTO"]] = estacion_mas_cercana
    distancia_meteo[fila["CODIGO_CORTO"]] = round(distancias.min(), 2)

print("Mapa de meteo asignada:")
print(mapa_meteo)
print()
print("Distancias (km) de cada emparejamiento de meteo:")
print(distancia_meteo)

Mapa de meteo asignada:
{4: 4, 8: 8, 16: 16, 18: 18, 24: 24, 35: 35, 36: 36, 38: 38, 39: 39, 54: 54, 56: 56, 58: 58, 59: 59, 11: np.int64(38), 17: np.int64(56), 27: np.int64(59), 40: np.int64(36), 47: np.int64(8), 48: np.int64(38), 49: np.int64(8), 50: np.int64(39), 55: np.int64(59), 57: np.int64(39), 60: np.int64(39)}

Distancias (km) de cada emparejamiento de meteo:
{4: 0.0, 8: 0.0, 16: 0.0, 18: 0.0, 24: 0.0, 35: 0.0, 36: 0.0, 38: 0.0, 39: 0.0, 54: 0.0, 56: 0.0, 58: 0.0, 59: 0.0, 11: np.float64(2.6), 17: np.float64(4.24), 27: np.float64(2.78), 40: np.float64(2.26), 47: np.float64(2.64), 48: np.float64(1.55), 49: np.float64(0.79), 50: np.float64(2.39), 55: np.float64(2.43), 57: np.float64(4.67), 60: np.float64(3.09)}


In [21]:
# Ahora buscamos el punto de trafico mas cercano para CADA estacion de aire (las 24)
# Tambien guardamos la distancia, para poder valorar despues la fiabilidad del emparejamiento
mapa_trafico = {}
distancia_trafico = {}

for _, fila in estaciones_aire.iterrows():
    distancias = haversine(fila["LONGITUD"], fila["LATITUD"], estaciones_trafico["longitud"], estaciones_trafico["latitud"])
    id_mas_cercano = estaciones_trafico.loc[distancias.idxmin(), "id"]
    mapa_trafico[fila["CODIGO_CORTO"]] = id_mas_cercano
    distancia_trafico[fila["CODIGO_CORTO"]] = round(distancias.min(), 3)

print("Mapa de trafico asignado:")
print(mapa_trafico)
print()
print("Distancias (km) de cada emparejamiento de trafico:")
print(distancia_trafico)

Mapa de trafico asignado:
{4: np.int64(4284), 8: np.int64(4028), 11: np.int64(3914), 16: np.int64(3791), 17: np.int64(4848), 18: np.int64(5058), 24: np.int64(3546), 27: np.int64(6928), 35: np.int64(3731), 36: np.int64(10890), 38: np.int64(3411), 39: np.int64(5416), 40: np.int64(5783), 47: np.int64(4129), 48: np.int64(4461), 49: np.int64(10080), 50: np.int64(5465), 54: np.int64(5365), 55: np.int64(6348), 56: np.int64(11007), 57: np.int64(9870), 58: np.int64(6547), 59: np.int64(9855), 60: np.int64(6573)}

Distancias (km) de cada emparejamiento de trafico:
{4: np.float64(0.06), 8: np.float64(0.075), 11: np.float64(0.052), 16: np.float64(0.08), 17: np.float64(0.213), 18: np.float64(0.488), 24: np.float64(1.182), 27: np.float64(0.168), 35: np.float64(0.03), 36: np.float64(0.03), 38: np.float64(0.068), 39: np.float64(0.049), 40: np.float64(0.133), 47: np.float64(0.152), 48: np.float64(0.088), 49: np.float64(0.433), 50: np.float64(0.149), 54: np.float64(0.377), 55: np.float64(0.47), 56: np.fl

**¿Qué acabamos de hacer?** Para cada una de las 24 estaciones de aire, hemos calculado "cuál es el punto de tráfico más cercano" y "cuál es la estación con meteo más cercana" (si no tenía la suya propia). El resultado son dos listas (`mapa_meteo` y `mapa_trafico`) tipo *"la estación X usa el tráfico del punto Y"*.

También guardamos la distancia de cada emparejamiento. Si algún día vemos una distancia muy grande (varios km), sabremos que ese dato es menos fiable, porque el punto de tráfico o meteo asignado no está realmente al lado de la estación de aire.

## 5. Unión final

Unimos NO2 (target) + meteo + trafico, usando los mapas de emparejamiento que acabamos de calcular.

In [22]:
# Añadimos las columnas de emparejamiento a la tabla de NO2
no2_largo["estacion_meteo"] = no2_largo["estacion"].map(mapa_meteo)
no2_largo["id_trafico"] = no2_largo["estacion"].map(mapa_trafico)

# Añadimos tambien las distancias, para poder valorar la fiabilidad del emparejamiento en el EDA
no2_largo["distancia_meteo_km"] = no2_largo["estacion"].map(distancia_meteo)
no2_largo["distancia_trafico_km"] = no2_largo["estacion"].map(distancia_trafico)

# Union con meteo (por estacion_meteo y fecha)
meteo_para_unir = meteo_diario.rename(columns={"estacion": "estacion_meteo"})
dataset = no2_largo.merge(meteo_para_unir, on=["estacion_meteo", "fecha"], how="left")

# Union con trafico (por id_trafico y fecha)
trafico_para_unir = trafico.rename(columns={"id_estacion_trafico": "id_trafico"})
dataset = dataset.merge(trafico_para_unir, on=["id_trafico", "fecha"], how="left")

print("Dataset unido:", dataset.shape)
dataset.head()

Dataset unido: (51001, 21)


,estacion,fecha,no2,estacion_meteo,id_trafico,distancia_meteo_km,distancia_trafico_km,humedad,precipitacion,temperatura,...,intensidad_mean,intensidad_max,ocupacion_mean,ocupacion_max,carga_mean,carga_max,vmed_mean,vmed_max,tipo_elem_moda,periodo_integracion_moda
0,4,2021-01-01,10,4,4284,0.0,0.06,NaN,NaN,4.070833,...,388.155556,916.0,3.522222,8.0,21.722222,50.0,0.0,0.0,URB,15.0
1,4,2021-01-02,23,4,4284,0.0,0.06,NaN,NaN,2.387500,...,516.344828,924.0,4.574713,8.0,28.563218,49.0,0.0,0.0,URB,15.0
2,4,2021-01-03,29,4,4284,0.0,0.06,NaN,NaN,2.233333,...,476.076923,924.0,4.109890,9.0,26.329670,50.0,0.0,0.0,URB,15.0
3,4,2021-01-04,29,4,4284,0.0,0.06,NaN,NaN,3.016667,...,677.088235,998.0,5.911765,8.0,37.044118,54.0,0.0,0.0,URB,15.0
4,4,2021-01-05,54,4,4284,0.0,0.06,NaN,NaN,0.250000,...,588.095745,922.0,4.978723,9.0,32.202128,49.0,0.0,0.0,URB,15.0


## 6. Nos quedamos con las columnas que nos interesan

In [23]:
columnas_utiles = [
    "estacion", "fecha", "no2",                                      
    "temperatura", "humedad", "viento_vel", "precipitacion",         
    "intensidad_mean", "intensidad_max",                              
    "ocupacion_mean", "carga_mean",
    "vmed_mean", "vmed_max",                                          
    "tipo_elem_moda",                                                 
    "distancia_meteo_km", "distancia_trafico_km"                      
]

df = dataset[columnas_utiles].copy()

print("Dataset final:", df.shape)
print()
print("Nulos por columna:")
print(df.isna().sum())
df.head()

Dataset final: (51001, 16)

Nulos por columna:
estacion                    0
fecha                       0
no2                         0
temperatura              5796
humedad                  4182
viento_vel              36316
precipitacion           27679
intensidad_mean         20832
intensidad_max          20832
ocupacion_mean          20849
carga_mean              20832
vmed_mean               20832
vmed_max                20832
tipo_elem_moda          20832
distancia_meteo_km          0
distancia_trafico_km        0
dtype: int64


,estacion,fecha,no2,temperatura,humedad,viento_vel,precipitacion,intensidad_mean,intensidad_max,ocupacion_mean,carga_mean,vmed_mean,vmed_max,tipo_elem_moda,distancia_meteo_km,distancia_trafico_km
0,4,2021-01-01,10,4.070833,NaN,NaN,NaN,388.155556,916.0,3.522222,21.722222,0.0,0.0,URB,0.0,0.06
1,4,2021-01-02,23,2.387500,NaN,NaN,NaN,516.344828,924.0,4.574713,28.563218,0.0,0.0,URB,0.0,0.06
2,4,2021-01-03,29,2.233333,NaN,NaN,NaN,476.076923,924.0,4.109890,26.329670,0.0,0.0,URB,0.0,0.06
3,4,2021-01-04,29,3.016667,NaN,NaN,NaN,677.088235,998.0,5.911765,37.044118,0.0,0.0,URB,0.0,0.06
4,4,2021-01-05,54,0.250000,NaN,NaN,NaN,588.095745,922.0,4.978723,32.202128,0.0,0.0,URB,0.0,0.06


**¿Qué acabamos de hacer?** De toda la tabla (que tenía columnas de sobra, como IDs internos), nos quedamos solo con las que vamos a usar de verdad: la fecha, la estación, el NO2 (target) y las features de meteo y tráfico.



## 7. Guardar el dataset final

In [24]:
df.to_parquet(ruta + "df.parquet", index=False)
print("Guardado en", ruta + "df.parquet")

Guardado en ../data_sample/df.parquet
